In [1]:
import torch
from transformers import AutoImageProcessor, AutoModelForImageClassification

### Load and inspect the model

In [2]:
MODEL_ID = "microsoft/swin-tiny-patch4-window7-224"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageClassification.from_pretrained(MODEL_ID)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


preprocessor_config.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/71.8k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  113MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/221 [00:00<?, ?it/s]

In [3]:
model.eval()

print("Device: ", device)
print("Model :", type(model).__name__)
print("Top-level modules :")
for name, module in model.named_children():
    print(name, "→", type(module).__name__)

Device:  cpu
Model : SwinForImageClassification
Top-level modules :
swin → SwinModel
classifier → Linear


**Configuration X-ray**

In [4]:
stage_dimensions = [
    model.config.embed_dim * (2 ** stage_index)
    for stage_index in range(len(model.config.depths))
]

print("Image size        :", model.config.image_size)
print("Patch size        :", model.config.patch_size)
print("Window size       :", model.config.window_size)
print("Initial dimension :", model.config.embed_dim)
print("Stage dimensions  :", stage_dimensions)
print("Stage depths      :", model.config.depths)
print("Attention heads   :", model.config.num_heads)
print("MLP ratio         :", model.config.mlp_ratio)
print("Number of classes :", model.config.num_labels)
print(
    "Absolute position embedding:",
    model.config.use_absolute_embeddings
)

Image size        : 224
Patch size        : 4
Window size       : 7
Initial dimension : 96
Stage dimensions  : [96, 192, 384, 768]
Stage depths      : [2, 2, 6, 2]
Attention heads   : [3, 6, 12, 24]
MLP ratio         : 4.0
Number of classes : 1000
Absolute position embedding: False


### Patch Embedding

**Image to patch token**

In [5]:
pixel_values = torch.randn(1, 3, 224, 224, device=device)

patch_embedding_layer = (model.swin.embeddings.patch_embeddings)

with torch.inference_mode():
    patch_tokens, grid_size = patch_embedding_layer(
        pixel_values
    )

print("Input image :", pixel_values.shape)
print("Patch grid :", grid_size)
print("Patch tokens :", patch_tokens.shape)
print("Projection Layer :", patch_embedding_layer.projection)    

Input image : torch.Size([1, 3, 224, 224])
Patch grid : (56, 56)
Patch tokens : torch.Size([1, 3136, 96])
Projection Layer : Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
